In [ ]:
import os
import json
import time
import requests
import pandas as pd
from urllib.parse import unquote

In [ ]:
#설정
RAW_KEY = ""
SERVICE_KEY = unquote(RAW_KEY)

BASE_URL = "https://apis.data.go.kr/4500000/busRouteInfo"
TIMEOUT = 60

In [ ]:
#공통함수
def call_api(operation, page_no=1, num_of_rows=100, **extra):
    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": page_no,
        "numOfRows": num_of_rows,
        "_type": "json",
        **extra,
    }
    res = requests.get(f"{BASE_URL}/{operation}", params=params, timeout=TIMEOUT)
    if res.status_code != 200:
        raise RuntimeError(f"{res.status_code}: {res.text[:300]}")
    try:
        return res.json()
    except ValueError:
        raise RuntimeError(f"JSON 아님: {res.text[:300]}")


def extract_rows(data):
    items = data.get("response", {}).get("body", {}).get("items", [])
    if isinstance(items, dict):
        items = items.get("item", items)
    if isinstance(items, dict):
        items = [items]
    return items or []


def fetch_all(operation, max_pages=50, num_of_rows=100, **extra):
    rows = []
    for page in range(1, max_pages + 1):
        data = call_api(operation, page_no=page, num_of_rows=num_of_rows, **extra)
        chunk = extract_rows(data)
        if not chunk:
            break
        rows.extend(chunk)
        total = data.get("response", {}).get("body", {}).get("totalCount")
        if total is not None and len(rows) >= int(total):
            break
        time.sleep(0.2)
    return pd.DataFrame(rows)

In [ ]:
#노선정보
if os.path.exists("routes.csv"):
    routes = pd.read_csv("routes.csv", dtype=str)
    print("routes.csv 불러옴:", routes.shape)
else:
    routes = fetch_all("getRouteInfoAll")
    routes.to_csv("routes.csv", index=False, encoding="utf-8-sig")
    print("노선 수집 완료:", routes.shape)

print(routes.head())

In [ ]:
#노선별 경유 정류소
def fetch_stops_by_route(route_ids, out="route_stops.jsonl"):
    done = set()
    if os.path.exists(out):
        with open(out, encoding="utf-8") as f:
            done = {json.loads(line)["route_id"] for line in f if line.strip()}
    print(f"이미 받은 노선: {len(done)}개")

    with open(out, "a", encoding="utf-8") as f:
        for i, rid in enumerate(route_ids, 1):
            if rid in done:
                continue
            try:
                data = call_api("getStaionByRoute", num_of_rows=200, busRouteId=rid)
                for row in extract_rows(data):
                    row["route_id"] = rid
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
            except Exception as e:
                print(f"[실패] {rid}: {e}")
            if i % 20 == 0:
                print(f"{i}/{len(route_ids)} 완료")
            time.sleep(0.3)


fetch_stops_by_route(routes["route_id"].tolist())

In [ ]:
#확인
stops = pd.read_json("route_stops.jsonl", lines=True, dtype=False)
print("정류소 데이터:", stops.shape)
print(stops.columns.tolist())
print(stops.head())

stops.to_csv("route_stops.csv", index=False, encoding="utf-8-sig")

print("노선당 최대 정류소 수:", stops.groupby("route_id").size().max())

# TAGO

In [ ]:
import os, json, time, requests
import pandas as pd
from urllib.parse import unquote

KEY = unquote("")
TAGO = "https://apis.data.go.kr/1613000/BusRouteInfoInqireService"
TARGETS = {"김천시": "김천", "고창군": "고창"}

In [ ]:
def call(base, op, **params):
    p = {"serviceKey": KEY, "_type": "json", **params}
    r = requests.get(f"{base}/{op}", params=p, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
    return r.json()


def rows_of(data):
    items = data.get("response", {}).get("body", {}).get("items", [])
    if isinstance(items, dict):
        items = items.get("item", [])
    if isinstance(items, dict):
        items = [items]
    return items or []


def fetch_all(base, op, per=100, max_pages=100, **params):
    out = []
    for pg in range(1, max_pages + 1):
        d = call(base, op, pageNo=pg, numOfRows=per, **params)
        ch = rows_of(d)
        if not ch:
            break
        out += ch
        tot = d["response"]["body"].get("totalCount")
        if tot is not None and len(out) >= int(tot):
            break
        time.sleep(0.2)
    return out


def pick(df, cands, label):
    low = {c.lower(): c for c in df.columns}
    for c in cands:
        if c in low:
            return low[c]
    raise KeyError(f"[{label}] 컬럼을 못 찾음. 실제 컬럼: {list(df.columns)}")

In [ ]:
#도시코드
cty = pd.DataFrame(rows_of(call(TAGO, "getCtyCodeList")))
print("도시코드 컬럼:", list(cty.columns), "/ 전체", len(cty), "개")
c_code = pick(cty, ["citycode"], "도시코드")
c_name = pick(cty, ["cityname"], "도시명")
city_codes = {}
for sigun, kw in TARGETS.items():
    hit = cty[cty[c_name].str.contains(kw)]
    print(sigun, "->", hit[[c_code, c_name]].values.tolist())
    if len(hit) == 1:
        city_codes[sigun] = str(hit.iloc[0][c_code])
print("사용할 도시코드:", city_codes)
assert len(city_codes) == len(TARGETS), "위 출력 보고 도시코드 직접 지정 필요"

In [ ]:
#TAGO노선목록
tago_routes = {}
for sigun, cc in city_codes.items():
    df = pd.DataFrame(fetch_all(TAGO, "getRouteNoList", cityCode=cc))
    print(sigun, "노선 행 수:", len(df), "/ 컬럼:", list(df.columns))
    tago_routes[sigun] = df

In [ ]:
#TAGO 노선별 경유 정류소
OUT = "tago_stops.jsonl"
done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as f:
        done = {(j["_sigun"], j["_rid"]) for j in map(json.loads, filter(str.strip, f))}

with open(OUT, "a", encoding="utf-8") as f:
    for sigun, df in tago_routes.items():
        rid_col = pick(df, ["routeid"], f"{sigun} routeId")
        ids = df[rid_col].dropna().astype(str).unique().tolist()
        for i, rid in enumerate(ids, 1):
            if (sigun, rid) in done:
                continue
            try:
                for row in fetch_all(TAGO, "getRouteAcctoThrghSttnList",
                                     cityCode=city_codes[sigun], routeId=rid):
                    row["_sigun"], row["_rid"] = sigun, rid
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
            except Exception as e:
                print(f"[실패] {sigun} {rid}: {e}")
            if i % 20 == 0:
                print(f"{sigun} {i}/{len(ids)}")
            time.sleep(0.2)

tago_stops = pd.read_json(OUT, lines=True, dtype=False)
print("TAGO 정류소 컬럼:", list(tago_stops.columns))

# 공주

In [ ]:
g_routes = pd.read_csv("routes.csv", dtype=str)
g_stops = pd.read_csv("route_stops.csv", dtype=str)
print("공주 정류소 컬럼:", list(g_stops.columns))

In [ ]:
#병합
def normalize(df, sigun, rid, cands):
    out = pd.DataFrame({"sigun": sigun})
    out = pd.DataFrame(index=df.index)
    out["sigun"] = sigun
    out["route_id"] = df[rid].astype(str)
    for name, (cs, label) in cands.items():
        out[name] = df[pick(df, cs, f"{sigun} {label}")]
    return out

COMMON = {
    "stop_seq":  (["nodeord", "stop_order", "station_order", "sta_order", "seq", "order", "stop_seq", "station_seq"], "순번"),
    "stop_id":   (["nodeid", "stop_id", "station_id", "sta_id"], "정류장 키"),
    "stop_name": (["nodenm", "stop_name", "station_name", "sta_name", "stop_nm", "station_nm"], "정류장명"),
    "lat":       (["gpslati", "lat", "latitude", "y", "stop_lat", "station_lat"], "위도"),
    "lon":       (["gpslong", "lon", "lng", "longitude", "x", "stop_lon", "station_lon"], "경도"),
}

parts = []

In [ ]:
# 공주
##오류발생!!
gj = normalize(g_stops, "공주시", pick(g_stops, ["route_id"], "공주 route_id"), COMMON)
nm = g_routes.set_index("route_id")
gj["route_no"] = gj["route_id"].map(nm["route_name"])
gj["first_time"] = gj["route_id"].map(nm["fst_time"])
gj["last_time"] = gj["route_id"].map(nm["lst_time"])
parts.append(gj)

In [ ]:
import requests
from urllib.parse import unquote

#실패건 확인
KEY = unquote("")
TAGO = "https://apis.data.go.kr/1613000/BusRouteInfoInqireService"
FAILED = [("김천시", "37030", "GCB47201478"), ("김천시", "37030", "GCB47201451"),
          ("김천시", "37030", "GCB47201781"), ("고창군", "35370", "TSB318000088"),
          ("고창군", "35370", "TSB318000089"), ("고창군", "35370", "TSB318000132"),
          ("고창군", "35370", "TSB318000238")]
for sg, cc, rid in FAILED[:2]:
    r = requests.get(f"{TAGO}/getRouteAcctoThrghSttnList",
                     params={"serviceKey": KEY, "_type": "json", "cityCode": cc,
                             "routeId": rid, "pageNo": 1, "numOfRows": 100}, timeout=60)
    print(sg, rid, r.status_code, r.text[:400], "\n")

In [ ]:
tago_stops = pd.read_json("tago_stops.jsonl", lines=True, dtype=False)
g_routes = pd.read_csv("routes.csv", dtype=str)
g_stops = pd.read_csv("route_stops.csv", dtype=str)

COLS = ["sigun", "route_id", "route_no", "stop_seq", "stop_id", "stop_name", "lat", "lon", "first_time", "last_time"]

In [ ]:
# 공주 확인
rt = g_routes.drop_duplicates("route_id").set_index("route_id")
gj = pd.DataFrame({
    "sigun": "공주시",
    "route_id": g_stops["route_id"],
    "route_no": g_stops["route_id"].map(rt["route_name"]),
    "stop_seq": g_stops["stop_ord"],
    "stop_id": g_stops["stop_id"],
    "stop_name": g_stops["stop_name"],
    "lat": g_stops["lat"],
    "lon": g_stops["lng"],
    "first_time": g_stops["route_id"].map(rt["fst_time"]),
    "last_time": g_stops["route_id"].map(rt["lst_time"]),
})

In [ ]:
# 김천, 고창 확인

no_map = {}
for sg, df in tago_routes.items():
    no_map[sg] = df.drop_duplicates("routeid").set_index("routeid")["routeno"].astype(str)

tg = pd.DataFrame({
    "sigun": tago_stops["_sigun"],
    "route_id": tago_stops["_rid"].astype(str),
    "stop_seq": tago_stops["nodeord"],
    "stop_id": tago_stops["nodeid"],
    "stop_name": tago_stops["nodenm"],
    "lat": tago_stops["gpslati"],
    "lon": tago_stops["gpslong"],
    "first_time": None,
    "last_time": None,
})
tg["route_no"] = [no_map[s].get(r) for s, r in zip(tg["sigun"], tg["route_id"])]

In [ ]:
# 합치기
final = pd.concat([gj[COLS], tg[COLS]], ignore_index=True)
for c in ["lat", "lon", "stop_seq"]:
    final[c] = pd.to_numeric(final[c], errors="coerce")
final.to_csv("routes_stops.csv", index=False, encoding="utf-8-sig")

print("\n=== 시군별 노선 개수 ===")
print(final.groupby("sigun")["route_id"].nunique())
print("\n=== 시군별 행 수 / 좌표 결측 ===")
print(final.groupby("sigun").agg(rows=("stop_id", "size"), lat_na=("lat", lambda s: s.isna().sum())))
print("\n=== routes_stops.csv 첫 10줄 ===")
print(final.head(10).to_string())

- 확인

In [ ]:
def rows_of(data):
    items = data.get("response", {}).get("body", {}).get("items", [])
    if not isinstance(items, (dict, list)):
        return []
    if isinstance(items, dict):
        items = items.get("item", [])
    if isinstance(items, dict):
        items = [items]
    return items if isinstance(items, list) else []

In [ ]:
import json, time
with open("tago_stops.jsonl", "a", encoding="utf-8") as f:
    for sg, cc, rid in FAILED:
        rows = fetch_all(TAGO, "getRouteAcctoThrghSttnList", cityCode=cc, routeId=rid)
        print(sg, rid, "->", len(rows), "건")
        for row in rows:
            row["_sigun"], row["_rid"] = sg, rid
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        time.sleep(0.2)